## Prescribe Velocity

In [ ]:
import json
import sys
from pathlib import Path
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "hamageolib").is_dir()
)
sys.path.insert(0, str(repo_root))

from hamageolib.utils.aspect.io import save_parameters_from_dict

## Create case

### Create a Stegman Case

In [ ]:
is_make_stegman_case = False

if is_make_stegman_case:

    from hamageolib.research.prescribe_vecloity.config import make_stegman_case
    
    # 2-D case
    case_dir = Path("/mnt/lochy/ASPECT_DATA/MOW/test_prescribe")/"stegman_2d_prescribed_velocity_0"
    case_dir.mkdir(parents=True, exist_ok=False)
    
    prm_dict, wb_dict, context = make_stegman_case({
          "end_time": 5e6,
          "adaptive_refinement": 2,
          "refine_plate_with_isosurfaces": True,
          "long_slab": True, # longer slab by 300 km length
          "prescribe_slab_velocity": True, # prescribed condition
          "slab_velocity_magnitude": 0.05 # prescribed condition, m/year
    })
    
    # 3-D case
    # case_dir = Path("/mnt/lochy/ASPECT_DATA/MOW/test_prescribe")/"stegman_3d_1"
    # case_dir.mkdir(parents=True, exist_ok=True)
    # prm_dict, wb_dict, context = make_stegman_case({
    #     "dimension": 3,
    #       "x_repetitions": 4,
    #       "y_repetitions": 2,
    #       "z_repetitions": 1,
    #       "end_time": 5e6,
    #       "graphical_output_interval": 5e5,
    #       "checkpoint_interval": 5e5,
    #       "adaptive_refinement": 2,
    #       "refine_plate_with_isosurfaces": True,
    #       "trench_width": 1200e3, # slab width
    #       "long_slab": True, # longer slab by 300 km length
    #       "prescribe_slab_velocity": True, # prescribed condition
    #       "slab_velocity_magnitude": 0.05, # prescribed condition, m/year
    # })
    
    
    with (case_dir / "case.prm").open("w") as prm_file:
        save_parameters_from_dict(prm_file, prm_dict)
    
    with (case_dir / "case.wb").open("w") as wb_file:
        json.dump(wb_dict, wb_file, indent=2)
        wb_file.write("\n")
    
    print("New case: %s" % case_dir)

### Generate a trench-retreat case

Create a new case from the spherical `_2` reference, moving the trench by a configurable distance while anchoring the original terminal slab segment. 

A benmark is set with `_3` for the case name, with 100e3-meter retreat. This feature is reproduced,including its 20 km overlap collars.

The Python utility samples WorldBuilder's Bezier trench directly; no compiled sampler is needed. It supports the east-dipping, four-segment spherical `begin segment` profile with constant thickness and equal transition lengths. It copies `case.prm` exactly and writes the new `case.wb`, `geometry.npz`, and `geometry_report.json`. 

It does not copy simulation outputs or execute ASPECT. New retreat distances need field/mesh validation before production use.

In [ ]:
# Enable only after choosing a new destination directory.
do_generate_retreat_case = True

retreat_source_case = Path(
    "/mnt/lochy/ASPECT_DATA/MOW/test_prescribe/"
    "3d_spherical_subduction_along_surface_velocity_2"
)
retreat_destination_case = retreat_source_case.parent / (
    "3d_spherical_subduction_along_surface_velocity_3_generated"
)
trench_retreat_m = 100e3  # Free parameter: metres (e.g. 50e3 or 100e3).

if do_generate_retreat_case:
    import json
    import sys

    from hamageolib.research.prescribe_vecloity.trench_retreat import generate_retreat_case

    retreat_report = generate_retreat_case(
        retreat_source_case,
        retreat_destination_case,
        trench_retreat_m=trench_retreat_m,
    )
    print(json.dumps(retreat_report, indent=2))
    print(f"Created case: {retreat_destination_case}")


## Prescribed-velocity visualization

view: green translucent plate (0.8–1.0), white domain velocity arrows, and blue slab velocity arrows. Edit the case, saved time, glyph scale and sample counts below, then enable the block. The image and a reusable ParaView state are saved under `img/pv_outputs`.

The runner loads `base.py` and the separate prescribed-velocity workflow at runtime; it does not concatenate or modify either source. `PVPYTHON` can select a ParaView Python executable. Run this block independently of the case-creation cell above.


In [ ]:
from pathlib import Path

do_prescribed_velocity_visualization = True
visualization_case_dir = Path(
    "/mnt/lochy/ASPECT_DATA/MOW/test_prescribe/"
    "3d_spherical_subduction_along_surface_velocity_3"
)
visualization_time = 0.0  # Years; choose a saved PVD timestep.
velocity_glyph_scale = 5e6
velocity_domain_samples = 5000
velocity_slab_samples = 500

if do_prescribed_velocity_visualization:
    import os
    import shutil
    import subprocess
    from pathlib import Path

    visual_repo_root = next(
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "hamageolib").is_dir()
    )
    visual_scripts = visual_repo_root / "scripts/paraview_scripts"
    solution_file = visualization_case_dir / "output/solution.pvd"
    if not solution_file.is_file():
        raise FileNotFoundError(solution_file)
    pvpython_executable = shutil.which(os.environ.get("PVPYTHON", "pvpython"))
    if pvpython_executable is None:
        raise FileNotFoundError("Set PVPYTHON to a ParaView pvpython executable")

    image_dir = visualization_case_dir / "img/pv_outputs"
    script_dir = visualization_case_dir / "paraview_scripts"
    image_dir.mkdir(parents=True, exist_ok=True)
    script_dir.mkdir(parents=True, exist_ok=True)
    image_file = image_dir / "prescribed_velocity.png"
    state_file = image_dir / "prescribed_velocity.pvsm"
    runner_file = script_dir / "run_prescribed_velocity.py"
    runner_lines = [
        "import os, runpy",
        "os.environ.setdefault('PV_INTERFACE_PATH', '')",
        f"base = runpy.run_path({str(visual_scripts / 'base.py')!r})",
        f"workflow = runpy.run_path({str(visual_scripts / 'PrescribedVelocity/prescribed_velocity.py')!r}, init_globals=base)",
        "workflow['plot_prescribed_velocity'](",
        f"    {str(solution_file)!r}, {str(image_file)!r},",
        f"    time={visualization_time!r}, glyph_scale={velocity_glyph_scale!r},",
        f"    domain_samples={velocity_domain_samples!r}, slab_samples={velocity_slab_samples!r},",
        f"    state_file={str(state_file)!r})",
    ]
    runner_file.write_text("\n".join(runner_lines) + "\n", encoding="utf-8")

    print("Saved paraview script: %s" % str(runner_file))
    # subprocess.run(
    #     [pvpython_executable, "--force-offscreen-rendering", str(runner_file)],
    #     check=True,
    # )
    # print(f"Saved image: {image_file}")
    # print(f"Saved ParaView state: {state_file}")
